In [2]:
# ========================
# PARTE 0: MONTAR DRIVE E INSTALAR DEPENDÊNCIAS
# ========================
from google.colab import drive
drive.mount('/content/drive')

!apt-get update > /dev/null
!apt install chromium-chromedriver -y > /dev/null || true
!cp /usr/lib/chromium-browser/chromedriver /usr/bin || true
!pip install selenium pandas openpyxl requests > /dev/null

# ========================
# PARTE 1: IMPORTS E CONFIGS
# ========================
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException
import pandas as pd
import os, time, requests, zipfile, re
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor

# Configurações do navegador
options = Options()
options.binary_location = "/usr/bin/chromium-browser"
options.add_argument("--headless")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--blink-settings=imagesEnabled=false")
driver = webdriver.Chrome(options=options)
wait = WebDriverWait(driver, 7)

# ========================
# PARTE 2: CONFIG ALGOLIA
# ========================
ALGOLIA_APP_ID = "1CF3ZT43ZU"
ALGOLIA_API_KEY = "150c68d1c61fc1835826a57a203dab72"
ALGOLIA_INDEX = "production_products"

headers_algolia = {
    "Content-Type": "application/json",
    "X-Algolia-Application-Id": ALGOLIA_APP_ID,
    "X-Algolia-API-Key": ALGOLIA_API_KEY,
}

termos_busca = ["pisos", "revestimentos", "porcelanatos"]

def buscar_links_por_categoria(termos, max_resultados=10):
    links = []
    for termo in termos:
        print(f"🔍 Buscando: {termo}")
        pagina = 0
        while len(links) < max_resultados:
            url = f"https://{ALGOLIA_APP_ID}-dsn.algolia.net/1/indexes/{ALGOLIA_INDEX}/query"
            payload = { "params": f"query={termo}&page={pagina}&hitsPerPage=250" }
            res = requests.post(url, headers=headers_algolia, json=payload)
            if res.status_code != 200:
                break
            data = res.json()
            hits = data.get("hits", [])
            if not hits:
                break
            for item in hits:
                if len(links) >= max_resultados:
                    break
                relative_url = item.get("url")
                if relative_url:
                    full_url = f"https://www.leroymerlin.com.br{relative_url}" if not relative_url.startswith("http") else relative_url
                    if full_url not in links:
                        links.append(full_url)
            pagina += 1
            if pagina >= data.get("nbPages", 0):
                break
        if links:
            break  # Para ao encontrar 10 de qualquer termo
    return links

print("🔍 Coletando links...")
todos_links = buscar_links_por_categoria(termos_busca, max_resultados=10)
print(f"✅ Total de links coletados: {len(todos_links)}\n")

# ========================
# PARTE 3: EXTRAÇÃO DE PRODUTO
# ========================
def extrair_dados_produto(url):
    driver.get(url)
    try:
        wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "h1")))
    except TimeoutException:
        return {'URL': url, 'Erro': 'Timeout'}

    dados = {'URL': url}
    dados['Nome'] = driver.find_element(By.CSS_SELECTOR, "h1").text if driver.find_elements(By.CSS_SELECTOR, "h1") else None
    dados['Código'] = url.split("_")[-1] if "_" in url else None

    # Preço
    preco_txt = None
    seletores = [
        '[data-cy="product-price"]',
        '[data-cy="product-price-integer"]',
        '[data-cy="product-price-sale"]',
        '.price__sales', '.value__integer', '.price',
        '.price__value', '.sales-price', '.main-price'
    ]
    for sel in seletores:
        elems = driver.find_elements(By.CSS_SELECTOR, sel)
        if elems:
            text = elems[0].text.strip()
            if text and text.lower() != "grátis":
                preco_txt = text
                break
    if not preco_txt:
        html = driver.page_source
        match = re.search(r"R\$ ?\d{1,3}(?:\.\d{3})*,\d{2}", html)
        if match:
            preco_txt = match.group()

    if preco_txt:
        try:
            preco_val = float(preco_txt.replace("R$", "").replace("/m²", "").replace(" ", "").replace(".", "").replace(",", "."))
            dados['Preço Promocional'] = f"{preco_val:.2f}".replace(".", ",")
        except:
            dados['Preço Promocional'] = preco_txt
    else:
        dados['Preço Promocional'] = "N/A"

    dados['Descrição'] = driver.find_element(By.CSS_SELECTOR, 'meta[name="description"]').get_attribute("content") if driver.find_elements(By.CSS_SELECTOR, 'meta[name="description"]') else None
    dados['Imagem'] = driver.find_element(By.CSS_SELECTOR, 'meta[property="og:image"]').get_attribute("content") if driver.find_elements(By.CSS_SELECTOR, 'meta[property="og:image"]') else "N/A"

    specs = {}
    for tr in driver.find_elements(By.CSS_SELECTOR, 'table tr'):
        try:
            k = tr.find_element(By.CSS_SELECTOR, 'th').text.strip()
            v = tr.find_element(By.CSS_SELECTOR, 'td').text.strip()
            specs[k] = v
        except:
            continue
    dados['Especificações'] = "\n".join(f"{k}: {v}" for k, v in specs.items())
    dados['Loja'] = "Leroy Merlin"
    return dados

# ========================
# PARTE 4: PREPARAÇÃO DE PASTAS E ARQUIVOS
# ========================
data_nome = datetime.now().strftime("%Y-%m-%d")
base_drive = f"/content/drive/MyDrive/WebScraping/FEITO/Leroy/COLETA_LEROY/{data_nome}"
pasta_imagens = "/content/drive/MyDrive/WebScraping/FEITO/Leroy/COLETA_LEROY/imagens"
os.makedirs(base_drive, exist_ok=True)
os.makedirs(pasta_imagens, exist_ok=True)

arquivo_csv = os.path.join(base_drive, f"leroymerelin_{data_nome}.csv")
arquivo_excel = os.path.join(base_drive, f"leroymerelin_{data_nome}.xlsx")

# Carrega CSV existente para evitar duplicações
df_existente = pd.read_csv(arquivo_csv, sep=";", encoding="utf-8", on_bad_lines='skip') if os.path.exists(arquivo_csv) else pd.DataFrame()
codigos_existentes = df_existente["Código"].tolist() if "Código" in df_existente.columns else []

# ========================
# PARTE 5: COLETAR PRODUTOS
# ========================
todos = []
for i, link in enumerate(todos_links, 1):
    try:
        dados = extrair_dados_produto(link)
        codigo = dados.get("Código", f"produto_{i}")
        if codigo in codigos_existentes:
            print(f"⏩ {codigo} já coletado, pulando.")
            continue

        caminho_img = os.path.join(pasta_imagens, f"{codigo}.png")
        status_img = "já existe" if os.path.exists(caminho_img) else "será baixada"
        print(f"🔗 {i}/{len(todos_links)} -> {link} | 🖼️ Imagem: {status_img}")
        todos.append(dados)
    except Exception as e:
        print("❌ Erro ao processar:", e)

# ========================
# PARTE 6: BAIXAR IMAGENS EM PARALELO
# ========================
def baixar_imagem(dados):
    codigo = dados.get("Código", "sem_codigo")
    url_img = dados.get("Imagem")
    caminho = os.path.join(pasta_imagens, f"{codigo}.png")

    if os.path.exists(caminho) or not url_img or url_img == "N/A":
        return f"⏩ {codigo}.png já existe ou não aplicável"

    try:
        resp = requests.get(url_img, headers={"User-Agent": "Mozilla/5.0"}, timeout=10, stream=True)
        if resp.status_code == 200:
            with open(caminho, "wb") as f:
                for cb in resp.iter_content(1024):
                    f.write(cb)
            return f"✅ Imagem baixada: {codigo}.png"
    except Exception as e:
        return f"⚠️ Erro ao baixar {codigo}: {e}"
    return f"❌ Falha para {codigo}"

print("⏬ Baixando imagens em paralelo...")
with ThreadPoolExecutor(max_workers=8) as executor:
    resultados = list(executor.map(baixar_imagem, todos))
    for r in resultados:
        print(r)

# ========================
# PARTE 7: SALVAR CSV, EXCEL E ZIP
# ========================
df_novos = pd.DataFrame(todos)
colunas = ['Código', 'Nome', 'Preço Promocional', 'Descrição', 'Especificações', 'Imagem', 'URL', 'Loja']
df_novos = df_novos[colunas]

df_final = pd.concat([df_existente, df_novos], ignore_index=True)
df_final.drop_duplicates(subset=["Código"], inplace=True)

df_final.to_csv(arquivo_csv, index=False, sep=";")
df_final.to_excel(arquivo_excel, index=False)

# Zipar imagens novas (opcional)
zipname = os.path.join(base_drive, f"imagens_leroymerelin_{data_nome}.zip")
with zipfile.ZipFile(zipname, "w", zipfile.ZIP_DEFLATED) as z:
    for fn in os.listdir(pasta_imagens):
        z.write(os.path.join(pasta_imagens, fn), fn)

print(f"\n✅ Excel salvo em: {arquivo_excel}")
print(f"✅ CSV salvo em: {arquivo_csv}")
print(f"📦 Imagens zipadas em: {zipname}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


cp: '/usr/lib/chromium-browser/chromedriver' and '/usr/bin/chromedriver' are the same file
🔍 Coletando links...
🔍 Buscando: pisos
✅ Total de links coletados: 10

🔗 1/10 -> https://www.leroymerlin.com.br/piso-ceramico-marmorizado-brilhante-borda-arredondada-interno-61,5x61,5cm-brisa-rosagres_91700182 | 🖼️ Imagem: já existe
🔗 2/10 -> https://www.leroymerlin.com.br/piso-ceramico-marmorizado-acetinado-borda-arredondada-interno-58x58cm-vpc-58051-viva-ceramica_92112181 | 🖼️ Imagem: já existe
🔗 3/10 -> https://www.leroymerlin.com.br/piso-ceramico-marmorizado-acetinado-borda-reta-interno-81x81cm-selene-grey-formigres-prime_92343524 | 🖼️ Imagem: já existe
🔗 4/10 -> https://www.ler